# Feature Engineering

This notebook transforms the raw NBA game log dataset into a modeling dataset.

The goal is to:
- load the raw full-league dataset
- apply rolling and trend-based feature engineering
- inspect the generated features
- save the processed dataset for model training

In [1]:
import pandas as pd
import sys
import os
import importlib

## Load feature engineering module

The project uses reusable feature generation logic stored in `src/features.py`.

In [2]:
sys.path.append(os.path.abspath(".."))

import src.features
importlib.reload(src.features)

<module 'src.features' from 'c:\\dysk\\CV — kopia\\nba-player-props-model\\src\\features.py'>

## Load raw dataset

In [3]:
df = pd.read_csv("../data/raw/nba_players_2023_24_raw.csv")
df.head()

,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG3M,FG3A,FTM,FTA,REB,AST,TOV,STL,BLK,PTS,PLUS_MINUS,PLAYER_NAME,PLAYER_ID
0,22300277,2023-12-01,DAL vs. MEM,L,4,0,1,0,1,1,2,1,0,0,0,0,1,4,A.J. Lawson,1630639
1,22300287,2023-12-02,DAL vs. OKC,L,19,4,10,3,7,1,2,0,2,0,0,1,12,9,A.J. Lawson,1630639
2,22301213,2023-12-06,DAL vs. UTA,W,7,2,2,0,0,0,0,1,0,0,0,0,4,7,A.J. Lawson,1630639
3,22301226,2023-12-08,DAL @ POR,W,7,1,3,0,2,0,0,1,0,0,0,0,2,-4,A.J. Lawson,1630639
4,22300299,2023-12-11,DAL @ MEM,W,14,2,7,0,4,0,0,1,1,1,0,0,4,1,A.J. Lawson,1630639


## Raw dataset size

This provides a baseline before transformation.

print("Raw rows:", df.shape[0])
print("Raw columns:", df.shape[1])
print("Raw players:", df["PLAYER_NAME"].nunique())

## Generate modeling dataset

The feature engineering step creates:
- rolling scoring averages
- rolling minutes and shot volume
- rolling variability
- trend features
- contextual variables such as home/away and rest days

In [4]:
model_df = src.features.prepare_model_dataset(df)
model_df.head()

,PLAYER_NAME,PLAYER_ID,GAME_DATE,MATCHUP,PTS,HOME,days_rest,pts_last3,pts_last5,pts_last10,...,fga_last3,fga_last5,fg3a_last5,fta_last5,reb_last5,ast_last5,pts_std_last5,pts_trend,min_trend,fga_trend
0,A.J. Lawson,1630639,2023-12-22,DAL @ HOU,8,0,2.0,2.000000,1.6,3.1,...,1.333333,1.0,0.4,0.4,0.4,0.0,2.607681,-1.5,1.866667,0.333333
1,A.J. Lawson,1630639,2023-12-23,DAL vs. SAS,17,1,1.0,4.666667,3.2,3.8,...,2.666667,1.8,0.6,0.8,0.8,0.2,3.633180,-0.6,4.600000,0.866667
2,A.J. Lawson,1630639,2023-12-25,DAL @ PHX,0,0,2.0,8.333333,6.2,4.3,...,4.666667,3.6,1.4,0.8,2.2,0.8,7.014271,1.9,4.133333,1.066667
3,A.J. Lawson,1630639,2024-01-03,DAL vs. POR,3,1,9.0,8.333333,6.2,3.9,...,5.333333,4.0,1.8,0.8,2.4,0.8,7.014271,2.3,4.666667,1.333333
4,A.J. Lawson,1630639,2024-01-05,DAL vs. POR,14,1,2.0,6.666667,5.6,4.0,...,5.333333,4.0,1.6,0.4,2.0,0.8,7.162402,1.6,1.066667,1.333333


## Processed dataset size

Compare the transformed dataset with the raw input.

In [5]:
print("Processed rows:", model_df.shape[0])
print("Processed columns:", model_df.shape[1])
print("Processed players:", model_df["PLAYER_NAME"].nunique())

Processed rows: 13201
Processed columns: 22
Processed players: 264


## Compare raw vs processed

Some rows are removed because rolling features require enough historical games for each player.

In [6]:
comparison_df = pd.DataFrame({
    "dataset": ["raw", "processed"],
    "rows": [df.shape[0], model_df.shape[0]],
    "columns": [df.shape[1], model_df.shape[1]],
    "players": [df["PLAYER_NAME"].nunique(), model_df["PLAYER_NAME"].nunique()]
})

comparison_df

,dataset,rows,columns,players
0,raw,15914,20,277
1,processed,13201,22,264


## Processed column names

Review the final variables available for modeling.

In [7]:
model_df.columns.tolist()

['PLAYER_NAME',
 'PLAYER_ID',
 'GAME_DATE',
 'MATCHUP',
 'PTS',
 'HOME',
 'days_rest',
 'pts_last3',
 'pts_last5',
 'pts_last10',
 'min_last3',
 'min_last5',
 'fga_last3',
 'fga_last5',
 'fg3a_last5',
 'fta_last5',
 'reb_last5',
 'ast_last5',
 'pts_std_last5',
 'pts_trend',
 'min_trend',
 'fga_trend']

## Preview selected engineered features

In [8]:
selected_cols = [
    "PLAYER_NAME",
    "GAME_DATE",
    "MATCHUP",
    "PTS",
    "pts_last3",
    "pts_last5",
    "pts_last10",
    "pts_trend",
    "min_last3",
    "min_last5",
    "min_trend",
    "fga_last3",
    "fga_last5",
    "fga_trend",
    "days_rest",
    "HOME"
]

model_df[selected_cols].head(10)

,PLAYER_NAME,GAME_DATE,MATCHUP,PTS,pts_last3,pts_last5,pts_last10,pts_trend,min_last3,min_last5,min_trend,fga_last3,fga_last5,fga_trend,days_rest,HOME
0,A.J. Lawson,2023-12-22,DAL @ HOU,8,2.000000,1.6,3.1,-1.5,5.666667,3.8,1.866667,1.333333,1.0,0.333333,2.0,0
1,A.J. Lawson,2023-12-23,DAL vs. SAS,17,4.666667,3.2,3.8,-0.6,13.000000,8.4,4.600000,2.666667,1.8,0.866667,1.0,1
2,A.J. Lawson,2023-12-25,DAL @ PHX,0,8.333333,6.2,4.3,1.9,18.333333,14.2,4.133333,4.666667,3.6,1.066667,2.0,0
3,A.J. Lawson,2024-01-03,DAL vs. POR,3,8.333333,6.2,3.9,2.3,19.666667,15.0,4.666667,5.333333,4.0,1.333333,9.0,1
4,A.J. Lawson,2024-01-05,DAL vs. POR,14,6.666667,5.6,4.0,1.6,14.666667,13.6,1.066667,5.333333,4.0,1.333333,2.0,1
5,A.J. Lawson,2024-01-09,DAL vs. MEM,3,5.666667,8.4,5.0,3.4,12.666667,18.4,-5.733333,4.666667,5.6,-0.933333,4.0,1
6,A.J. Lawson,2024-01-13,DAL vs. NOP,3,6.666667,7.4,5.3,2.1,12.666667,14.8,-2.133333,5.000000,5.4,-0.400000,4.0,1
7,A.J. Lawson,2024-01-15,DAL vs. NOP,3,6.666667,4.6,5.4,-0.8,12.000000,9.8,2.200000,4.666667,4.0,0.666667,2.0,1
8,A.J. Lawson,2024-01-17,DAL @ LAL,2,3.000000,5.2,5.7,-0.5,8.333333,11.6,-3.266667,3.000000,4.2,-1.200000,2.0,0
9,A.J. Lawson,2024-01-24,DAL vs. PHX,3,2.666667,5.0,5.3,-0.3,8.333333,11.0,-2.666667,2.333333,3.6,-1.266667,7.0,1


## Missing values check after feature engineering

The processed dataset should be ready for training with no missing values in modeling columns.

In [9]:
model_df.isna().sum().sort_values(ascending=False)

PLAYER_NAME      0
PLAYER_ID        0
GAME_DATE        0
MATCHUP          0
PTS              0
HOME             0
days_rest        0
pts_last3        0
pts_last5        0
pts_last10       0
min_last3        0
min_last5        0
fga_last3        0
fga_last5        0
fg3a_last5       0
fta_last5        0
reb_last5        0
ast_last5        0
pts_std_last5    0
pts_trend        0
min_trend        0
fga_trend        0
dtype: int64

## Descriptive statistics for selected features

In [10]:
feature_cols = [
    "pts_last3",
    "pts_last5",
    "pts_last10",
    "pts_trend",
    "min_last3",
    "min_last5",
    "min_trend",
    "fga_last3",
    "fga_last5",
    "fga_trend",
    "days_rest",
    "pts_std_last5"
]

model_df[feature_cols].describe()

,pts_last3,pts_last5,pts_last10,pts_trend,min_last3,min_last5,min_trend,fga_last3,fga_last5,fga_trend,days_rest,pts_std_last5
count,13201.000000,13201.000000,13201.00000,13201.000000,13201.000000,13201.000000,13201.000000,13201.000000,13201.000000,13201.000000,13201.000000,13201.000000
mean,12.473070,12.451254,12.40203,0.049224,25.531096,25.480494,0.050602,9.548873,9.531884,0.016989,2.646163,5.371194
std,7.864618,7.539839,7.26967,2.036261,9.032574,8.741746,2.193750,5.606071,5.445977,1.325465,3.075763,2.617375
min,0.000000,0.000000,0.20000,-9.400000,0.333333,1.000000,-10.866667,0.000000,0.000000,-7.600000,1.000000,0.000000
25%,6.333333,6.800000,6.90000,-1.200000,19.666667,19.600000,-1.266667,5.333333,5.400000,-0.800000,2.000000,3.492850
50%,11.000000,11.200000,10.90000,0.100000,27.000000,27.000000,0.000000,8.666667,8.600000,0.066667,2.000000,4.979960
75%,17.333333,17.000000,16.60000,1.300000,32.666667,32.400000,1.400000,13.000000,13.000000,0.866667,3.000000,6.804410
max,50.666667,44.200000,40.40000,9.400000,46.333333,44.600000,11.733333,30.333333,27.600000,6.200000,110.000000,21.384574


## Example feature history for one player

This helps verify whether rolling and trend features look reasonable for an individual player.

In [13]:
sample_player = model_df["PLAYER_NAME"].value_counts().index[0]

print(f"Example feature history for: {sample_player}")

model_df[model_df["PLAYER_NAME"] == sample_player][
    ["GAME_DATE", "PTS", "pts_last5", "pts_last10", "pts_trend", "min_last5", "fga_last5"]
].head(12)

Example feature history for: Buddy Hield


,GAME_DATE,PTS,pts_last5,pts_last10,pts_trend,min_last5,fga_last5
1754,2023-11-14,2,15.0,13.1,1.9,22.4,13.4
1755,2023-11-19,3,11.6,11.9,-0.3,23.0,11.4
1756,2023-11-21,24,8.4,11.2,-2.8,21.8,10.6
1757,2023-11-22,31,11.2,12.5,-1.3,23.6,9.6
1758,2023-11-24,18,15.2,14.9,0.3,26.2,10.2
1759,2023-11-27,9,15.6,15.3,0.3,27.6,9.6
1760,2023-11-30,12,17.0,14.3,2.7,26.6,10.6
1761,2023-12-02,7,18.8,13.6,5.2,29.8,13.0
1762,2023-12-04,21,15.4,13.3,2.1,28.4,12.2
1763,2023-12-07,8,13.4,14.3,-0.9,27.8,11.0


## Save processed dataset

This file is used later in:
- model training
- evaluation
- prediction scripts
- Streamlit application

In [14]:
model_df.to_csv("../data/processed/nba_players_model_dataset.csv", index=False)
print("Saved to ../data/processed/nba_players_model_dataset.csv")

Saved to ../data/processed/nba_players_model_dataset.csv


## Observations

At this stage:
- raw game logs have been transformed into a player-level modeling dataset
- rolling and trend features are available
- rows without enough historical context were removed
- the processed dataset is ready for training and evaluation